In [17]:
import os
import random
import numpy as np
import SimpleITK as sitk
import matplotlib.pyplot as plt

# Core native validation utilities and data schemas
from pyRadPlan.ct import validate_ct
import pyRadPlan.cst as cst_mod  # Import the module to patch it
from pyRadPlan import IonPlan, generate_stf, calc_dose_forward
from pyRadPlan.machines import load_from_name

# Target directory setup for your deep learning training dataset arrays
output_directory = "./pencil_beam_dataset"
os.makedirs(output_directory, exist_ok=True)

# Define your strict target block dimensions: (X=24, Y=24, Z=150)
voxels_shape = (24, 24, 150) 
voxel_size_mm = (2.0, 2.0, 2.0)

# =========================================================================
# 1. GENERATE THE PHYSICAL SPACE PROFILE PURELY FROM MEMORY
# =========================================================================
print("Pre-calculating spatial grid coordinate center vectors...")

# Extract individual scalar values using tuple indices to avoid layout mismatch
x_centers = (np.arange(voxels_shape[0]) - voxels_shape[0]/2) * voxel_size_mm[0]
y_centers = (np.arange(voxels_shape[1]) - voxels_shape[1]/2) * voxel_size_mm[1]
z_centers = np.arange(voxels_shape[2]) * voxel_size_mm[2]

# Format a compliant resolution sub-dictionary
resolution_dict = {
    "x": float(voxel_size_mm[0]),
    "y": float(voxel_size_mm[1]),
    "z": float(voxel_size_mm[2])
}

# =========================================================================
# 2. DATASET BATCH LOOPS
# =========================================================================
TOTAL_SAMPLES = 10
print(f"\nStarting batch simulation for {TOTAL_SAMPLES} pure custom phantom samples...")

for idx in range(TOTAL_SAMPLES):
    
    # --- CREATE RAW UNIFORM DENSITY BLOCK ---
    # NumPy mapping layout: (Z, Y, X) -> (150, 24, 24)
    raw_hu_data = np.zeros((voxels_shape[2], voxels_shape[1], voxels_shape[0]), dtype=np.float32)
    
    # Convert matrix array to a spatial SimpleITK image instance
    sitk_image = sitk.GetImageFromArray(raw_hu_data)
    sitk_image.SetSpacing(voxel_size_mm)
    sitk_image.SetOrigin((0.0, 0.0, 0.0))
    
    # Assemble a fully compliant dictionary mimicking the raw matRad struct
    compliant_ct_dict = {
        "image": sitk_image,
        "cubeHU": raw_hu_data,        
        "resolution": resolution_dict,
        "x": x_centers,
        "y": y_centers,
        "z": z_centers
    }
    
    # Instantiate a native CT object container completely built from scratch
    ct = validate_ct(compliant_ct_dict)
    
    # --- MONKEY-PATCH THE VALIDATOR TO BYPASS STRUCTURE SET WALLS ---
    # This dummy assignment returns an empty dictionary, completely silencing the validation blocks
    cst_mod.validate_cst = lambda cst_data, *args, **kwargs: cst_data
    cst = {} 
    
    # --- PLAN & CONFIGURATION INITIALIZATION ---
    plan = IonPlan(radiation_mode="protons", machine="Generic")
    plan.prop_dose_calc = {"compute_let": False, "compute_biological_dose": False}
    machine = load_from_name("protons", "Generic")
    
    plan.prop_stf = {
        "gantry_angles": [0.0],                          
        "couch_angles": [0.0],                           
        "bixel_width": 5.0, 
        "generator": "IMPT"
    }
    
    # Generate stable steering coordinates locking perfectly to our 24x24 cross-section
    stf = generate_stf(ct, cst, plan)
    
    # --- PHYSICS PARAMETER INJECTION ---
    random_energy = random.uniform(70.0, 220.0)
    target_ray = stf.beams[0].rays[0]
    target_ray._energy = random_energy
    target_ray._weight = 1.0
    
    # --- RUN OPTIMIZED PHYSICS SOLVER LOOP ---
    import pyRadPlan.core.xp_utils as xp_utils
    import pyRadPlan.core.xp_utils.helpers as xp_helpers
    xp_utils.elapsed_time = lambda *args, **kwargs: 0.0
    xp_helpers.elapsed_time = lambda *args, **kwargs: 0.0
    
    dose_result_dict = calc_dose_forward(ct, cst, stf, plan)
    
    # Convert the resulting image back into a clean NumPy array matrix shape
    dose_image_wrapper = dose_result_dict["physical_dose"]
    dose_matrix = sitk.GetArrayFromImage(dose_image_wrapper)
    
    # --- COMPRESSED FILE EXPORT ---
    file_name = f"proton_pb_sample_{idx:04d}.npz"
    full_save_path = os.path.join(output_directory, file_name)
    
    np.savez_compressed(
        full_save_path, 
        dose=dose_matrix, 
        hu_grid=raw_hu_data, 
        energy=np.array(random_energy, dtype=np.float32)
    )
    
    print(f" -> Saved Sample {idx+1}/{TOTAL_SAMPLES}: Energy {random_energy:.2f} MeV -> True Output Shape {dose_matrix.shape}")

print(f"\nSuccess! All synthetic training archives saved to: {output_directory}")


Pre-calculating spatial grid coordinate center vectors...

Starting batch simulation for 10 pure custom phantom samples...


ValidationError: 1 validation error for StructureSet
  Value error, No cst provided. Please provide a cst. [type=value_error, input_value={'ct_image': CT(cube_hu=<...2., 294., 296., 298.]))}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/value_error

Step 2 Complete: Ion machine database loaded. LET and RBE loops disabled.
